# Time-domain scattering of a plane wave by two disks

In this notebook we use the broken-FEEC (CONGA) interface of PSYDAC to solve a time-domain Maxwell scattering problem,
and compare it with the frequency-domain solution of the "ExNonPenetrable" example of
[mu-diff](https://github.com/mu-diff/mu-diff) (`Examples/Solver/ExNonPenetrable.m`):
a plane wave $u_{\rm inc} = e^{i k\, \boldsymbol{d}\cdot\boldsymbol{x}}$ is scattered by two disks, and the total field
$u_{\rm inc} + u$ satisfies a Neumann (sound-hard) or Dirichlet (sound-soft) condition on the disks.

In the 2D TE setting, $u = B_z$ solves the wave equation, and the two mu-diff problems correspond to two kinds of
obstacles for Maxwell's equations, selected with `bc_type`:

- `'pec'`, a perfect electric conductor: $\boldsymbol{n}\times\boldsymbol{E} = 0$ for the total field on the disks.
  This is mu-diff's `NeumannSolver` (sound-hard disks, $\partial_n u = 0$), but for the electric field it is a standard
  homogeneous Dirichlet condition, imposed strongly on its tangential trace.
- `'pmc'`, a perfect magnetic conductor: $B = 0$ for the total field on the disks.
  This is mu-diff's `DirichletSolver` (sound-soft disks, $u = 0$), while for the electric field it is a natural
  condition, imposed weakly.

We solve for the scattered field $(\boldsymbol{E}, B) \in H(\mathrm{curl}) \times L^2$ on a box $\Omega$ with two circular holes:

$$
\begin{aligned}
\partial_t \boldsymbol{E} - \mathbf{curl}\, B &= 0 && \text{in } \Omega, \\
\partial_t B + \mathrm{curl}\, \boldsymbol{E} &= 0 && \text{in } \Omega, \\
B - \boldsymbol{n}\times\boldsymbol{E} &= 0 && \text{on the box (Silver-Müller absorbing condition)}, \\
\boldsymbol{n}\times\boldsymbol{E} = -\boldsymbol{n}\times\boldsymbol{E}_{\rm inc}(t)
\quad &\text{or} \quad B = -B_{\rm inc}(t) && \text{on the disks},
\end{aligned}
$$

with the incident plane wave $B_{\rm inc} = \mathrm{Re}\big(e^{i (k \boldsymbol{d}\cdot\boldsymbol{x} - \omega t)}\big)$, $\omega = k$,
entering the box with a smooth front. Once the solution is time-periodic, its complex amplitude $\hat B$ (such that
$B = \mathrm{Re}(\hat B e^{-i\omega t})$) approximates the scattered field $u$ computed by mu-diff.

**Why the incident wave only appears on the disks.** The unknowns are the *scattered* fields: the total field is
$(\boldsymbol{E}_{\rm inc} + \boldsymbol{E}, B_{\rm inc} + B)$, where the incident wave is known analytically everywhere
in the box. The physical boundary condition on the disks applies to the total field, e.g. $\boldsymbol{n}\times(\boldsymbol{E}_{\rm inc} + \boldsymbol{E}) = 0$,
which gives the inhomogeneous conditions above. On the box, only outgoing scattered waves need to be absorbed, so the
Silver-Müller condition is homogeneous: the incident wave does not have to be injected through the right boundary, and
is never propagated numerically.

This *scattered-field formulation* is equivalent to a *total-field formulation*, where the unknown is the total field,
the conditions on the disks are homogeneous, and the incident wave enters through an inhomogeneous Silver-Müller condition
$B - \boldsymbol{n}\times\boldsymbol{E} = B_{\rm inc} - \boldsymbol{n}\times\boldsymbol{E}_{\rm inc} = (1 - \boldsymbol{n}\cdot\boldsymbol{d})\, B_{\rm inc}$
on the box (that is $2B_{\rm inc}$ on the right side, $0$ on the left side and $B_{\rm inc}$ on the top and bottom sides).
Both give the same exact solution, but the scattered-field formulation avoids the numerical dispersion of the incident wave
across the box. It is also the formulation of mu-diff's integral equations, where the incident wave only enters as data on the obstacles.

To keep the notebook fast we use a coarse discretization and $k = \pi$ (mu-diff uses $k = 2\pi$).

In [ ]:
import numpy as np

k         = np.pi                  # wavenumber (omega = k, c = 1)
theta_inc = np.pi                  # incidence angle, as in mu-diff: d = (-1, 0)
bc_type   = 'pec'                  # 'pec' (mu-diff's NeumannSolver) or 'pmc' (mu-diff's DirichletSolver)
geometry  = 'analytic'             # 'analytic' (exact mappings) or 'spline' (their spline interpolation)

# the two disks of mu-diff's ExNonPenetrable, and the computational box
obstacles = [dict(center=(-2.5, -2.), radius=1.5),
             dict(center=( 2.5,  1.), radius=0.75)]
box = ((-6., 6.), (-6., 6.))

h      = 0.5                       # target mesh size (the wavelength is 2*pi/k = 2)
degree = [3, 3]

omega  = k
period = 2 * np.pi / omega
d      = np.array([np.cos(theta_inc), np.sin(theta_inc)])

## Step 1: Build the multipatch domain

The box is split into a tensor grid of patches. Each disk replaces exactly one cell of the grid, the square inscribed in the
circle, and the 4 cells sharing a side with it get a curved side on the circle, with the mapping below.
All the other cells are Cartesian patches.

Keeping every logical axis aligned with a Cartesian direction is what makes this layout work: SymPDE requires both
sides of an interface to use the same logical axis, and the conforming projections require interfaces with orientation $+1$.

In [ ]:
from sympde.topology import Square, Domain, IdentityMapping, AnalyticMapping

class DiskNeighbourMapping(AnalyticMapping):
    """
    Patch with one side on a circle (center (c1, c2), radius r0, angle from th0 to th1) and the opposite side on
    the segment from (c1+sx0, c2+sy0) to (c1+sx1, c2+sy1): linear blend in S = s0 + s1*x1 + s2*x2, both curves
    being parametrized by T = t1*x1 + t2*x2.
    """
    _expressions = {
        'x': 'c1 + (1-(s0+s1*x1+s2*x2))*r0*cos(th0 + (th1-th0)*(t1*x1+t2*x2))'
             ' + (s0+s1*x1+s2*x2)*(sx0 + (sx1-sx0)*(t1*x1+t2*x2))',
        'y': 'c2 + (1-(s0+s1*x1+s2*x2))*r0*sin(th0 + (th1-th0)*(t1*x1+t2*x2))'
             ' + (s0+s1*x1+s2*x2)*(sy0 + (sy1-sy0)*(t1*x1+t2*x2))'}
    _ldim = 2
    _pdim = 2

# grid breakpoints: box bounds and the inscribed squares of the disks
cells = [((cx - r/np.sqrt(2), cx + r/np.sqrt(2)), (cy - r/np.sqrt(2), cy + r/np.sqrt(2)))
         for (cx, cy), r in [(o['center'], o['radius']) for o in obstacles]]
xb = sorted({*box[0]} | {v for c in cells for v in c[0]})
yb = sorted({*box[1]} | {v for c in cells for v in c[1]})
obst_cells = [(xb.index(c[0][0]), yb.index(c[1][0])) for c in cells]

# the 4 neighbours of each disk: S, T (as coefficients), arc angles, and the face lying on the circle
q = np.pi / 4
sides = {
    'right':  (dict(s0=0, s1=1,  s2=0,  t1=0, t2=1), (-q,  q),   (0, -1)),
    'left':   (dict(s0=1, s1=-1, s2=0,  t1=0, t2=1), (5*q, 3*q), (0, +1)),
    'top':    (dict(s0=0, s1=0,  s2=1,  t1=1, t2=0), (3*q, q),   (1, -1)),
    'bottom': (dict(s0=1, s1=0,  s2=-1, t1=1, t2=0), (5*q, 7*q), (1, +1)),
}
neighbours = {}
for o, (i, j) in zip(obstacles, obst_cells):
    for side, cell in [('right', (i+1, j)), ('left', (i-1, j)), ('top', (i, j+1)), ('bottom', (i, j-1))]:
        neighbours[cell] = (o, side)

patches, obst_faces, squares = {}, [], {}
neighbours_of_face = {}                                  # patch name -> (center, radius) of its disk
for i in range(len(xb) - 1):
    for j in range(len(yb) - 1):
        if (i, j) in obst_cells:
            continue
        name = f'P_{i}_{j}'
        (x0, x1), (y0, y1) = (xb[i], xb[i+1]), (yb[j], yb[j+1])
        if (i, j) in neighbours:
            o, side = neighbours[i, j]
            coefs, (th0, th1), face = sides[side]
            (c1, c2), r0 = o['center'], o['radius']
            # far side of the cell, opposite to the disk
            sx0, sx1 = {'right': (x1, x1), 'left': (x0, x0)}.get(side, (x0, x1))
            sy0, sy1 = {'top': (y1, y1), 'bottom': (y0, y0)}.get(side, (y0, y1))
            F = DiskNeighbourMapping(f'M_{name}', 2, c1=c1, c2=c2, r0=r0, th0=th0, th1=th1,
                                     sx0=sx0-c1, sx1=sx1-c1, sy0=sy0-c2, sy1=sy1-c2, **coefs)
            squares[name] = Square(name)
            patches[i, j] = F(squares[name])
            obst_faces.append((name, *face))
            neighbours_of_face[name] = (o['center'], o['radius'])
        else:
            squares[name] = Square(name, bounds1=(x0, x1), bounds2=(y0, y1))
            patches[i, j] = IdentityMapping(f'M_{name}', 2)(squares[name])

# all interfaces join the same axis, with orientation +1
connectivity = []
for (i, j), P in patches.items():
    if (i+1, j) in patches:
        connectivity.append([(P, 0, +1), (patches[i+1, j], 0, -1), 1])
    if (i, j+1) in patches:
        connectivity.append([(P, 1, +1), (patches[i, j+1], 1, -1), 1])

domain = Domain.join(list(patches.values()), connectivity, name='domain')

obstacle_boundaries = [b for b in domain.boundary if (b.domain.logical_domain.name, b.axis, b.ext) in obst_faces]
outer_boundaries    = [b for b in domain.boundary if b not in obstacle_boundaries]
print(f'{len(domain.interior)} patches, {len(obstacle_boundaries)} faces on the disks, {len(outer_boundaries)} on the box')

## Step 2: Discretize the de Rham sequence

The number of cells is chosen per column and per row of the grid, so that it matches across interfaces.
PSYDAC also requires the cell counts of neighbouring patches to be multiples of each other in both directions,
hence we use powers of 2.

With `geometry = 'spline'`, each patch mapping is replaced by its interpolation in the spline space used for the
fields (same cells and degree: an *isoparametric* geometry), as would be the case for a geometry coming from a CAD file.
The discrete domain then takes its cells and degree from these spline mappings. The circles are no longer exact,
but the interfaces stay conforming: they are straight segments, which the spline interpolation reproduces exactly.

In [ ]:
from sympde.topology import Derham
from psydac.api.discretization import discretize

n_x = [2**int(np.ceil(np.log2((b - a) / h))) for a, b in zip(xb[:-1], xb[1:])]
n_y = [2**int(np.ceil(np.log2((b - a) / h))) for a, b in zip(yb[:-1], yb[1:])]
ncells = {}
for P in domain.interior:
    _, i, j = P.logical_domain.name.split('_')
    ncells[P.name] = [n_x[int(i)], n_y[int(j)]]

if geometry == 'spline':
    from psydac.mapping.discrete import SplineCallableMapping

    # interpolate each patch mapping in the spline space of the fields, and rebuild the domain with the same topology
    spline_patches = []
    for P in domain.interior:
        logical = P.logical_domain
        F_h = SplineCallableMapping.from_mapping(P.mapping, ncells=ncells[P.name], degree=degree,
                                                 bounds=zip(logical.min_coords, logical.max_coords))
        spline_patches.append(F_h.to_defined_mapping(f'S_{logical.name}')(squares[logical.name]))

    index = {P: n for n, P in enumerate(domain.interior)}
    spline_connectivity = [((index[I.minus.domain], I.minus.axis, I.minus.ext),
                            (index[I.plus.domain],  I.plus.axis,  I.plus.ext), I.ornt) for I in domain.interfaces]
    domain = Domain.join(spline_patches, spline_connectivity, name='domain')

    obstacle_boundaries = [b for b in domain.boundary if (b.domain.logical_domain.name, b.axis, b.ext) in obst_faces]
    outer_boundaries    = [b for b in domain.boundary if b not in obstacle_boundaries]

    # geometric error: distance between the spline boundaries of the disks and the true circles
    t = np.linspace(0, 1, 101)
    dist = 0
    for b in obstacle_boundaries:
        (cx, cy), r = neighbours_of_face[b.domain.logical_domain.name]
        eta = np.empty((2, t.size))
        eta[b.axis] = 0 if b.ext == -1 else 1
        eta[1 - b.axis] = t
        x, y = b.domain.mapping(*eta)
        dist = max(dist, abs(np.hypot(x - cx, y - cy) - r).max())
    print(f'max distance between the spline boundaries and the circles: {dist:.2e}')

    # the cells and degree come from the spline mappings
    domain_h = discretize(domain)
else:
    domain_h = discretize(domain, ncells=ncells)

derham   = Derham(domain, ["H1", "Hcurl", "L2"])
derham_h = discretize(derham, domain_h, degree=degree)

V0h, V1h, V2h = derham_h.spaces
print('dim(V1h) =', V1h.nbasis, ', dim(V2h) =', V2h.nbasis)

We plot the actual mesh: patch boundaries in black and cell edges in grey.

In [ ]:
from collections import OrderedDict
import matplotlib.pyplot as plt
from psydac.fem.plotting_utilities import get_patch_knots_gridlines

mappings = OrderedDict([(P.logical_domain, P.mapping) for P in domain.interior])
mappings_list = list(mappings.values())

fig, ax = plt.subplots(figsize=(6, 6))
for p in range(len(domain.interior)):
    for x, y in get_patch_knots_gridlines(V0h, 10, mappings, plotted_patch=p):
        ax.plot(x, y, color='0.6', lw=0.5)
        ax.plot(x[:, [0, -1]], y[:, [0, -1]], color='k', lw=1.2)
for o in obstacles:
    ax.add_patch(plt.Circle(o['center'], o['radius'], fill=False, color='k', lw=1.5))
ax.set_aspect('equal')
ax.set_title(f'Mesh: {len(domain.interior)} patches')
plt.show()

## Step 3: Obtain the FEEC operators

We need the Hodge (mass) operators, the broken curl $D^1$, the conforming projection $P^1$ and the commuting
projection on $V^1_h$ (for the incident field).
Only $P^1$ is needed among the conforming projections, so we build it directly.

In [ ]:
from psydac.feec.conforming_projectors import ConformingProjectionV1

nquads = [4 * (p + 1) for p in degree]
P0, P1, P2 = derham_h.projectors(nquads=nquads)

H0, H1, H2 = derham_h.hodge_operators(kind='linop')
bD0, bD1   = derham_h.derivatives(kind='linop')
cP1        = ConformingProjectionV1(V1h, hom_bc=False).linop

## Step 4: Absorbing boundary condition

The first-order Silver-Müller condition $B = \boldsymbol{n}\times\boldsymbol{E}$ on the box adds the boundary term
$$
\int_{\partial\Omega_{\rm box}} (\boldsymbol{n}\times\boldsymbol{E})(\boldsymbol{n}\times\boldsymbol{v})
$$
to the weak Ampère law.

In [ ]:
from sympde.calculus  import cross
from sympde.topology  import elements_of, NormalVector, Union
from sympde.expr.expr import integral, BilinearForm, LinearForm
from psydac.api.settings import PSYDAC_BACKENDS

backend = PSYDAC_BACKENDS['pyccel-gcc']

u, v = elements_of(derham.V1, names='u, v')
nn   = NormalVector('nn')
a    = BilinearForm((u, v), integral(Union(*outer_boundaries), cross(nn, u) * cross(nn, v)))
A_eps = discretize(a, domain_h, [V1h, V1h], backend=backend).assemble()

## Step 5: Sparse matrices and boundary conditions on the disks

The time loop is run on SciPy sparse matrices: the PSYDAC operators are assembled once, and applying their global
matrices at each time step is much cheaper.

- **PEC** is an essential condition on the tangential trace of $\boldsymbol{E}$: the conforming projection
  $P^1_{\rm hom}$ sets the tangential dofs on the disks to zero, and the incident data is added with a lift
  $\boldsymbol{E}_{\rm lift}(t)$ supported on these dofs, so that the electric field is $P^1_{\rm hom} \boldsymbol{E}_h + \boldsymbol{E}_{\rm lift}(t)$.
- **PMC** is a natural condition, imposed with the boundary term $+\int_{\rm disks} B_{\rm inc}\, (\boldsymbol{n}\times\boldsymbol{v})$ in the weak Ampère law.

In [ ]:
from scipy.sparse import diags

H1_m = H1.tosparse().tocsr()
H2_m = H2.tosparse().tocsr()
A_m  = A_eps.tosparse().tocsr()
D1_m = bD1.tosparse().tocsr()

def tangential_boundary_dofs(Vh, boundaries):
    """
    Global indices of the dofs of Vh carrying the tangential trace on the given boundaries,
    and their physical locations (images of their Greville points on the faces).
    """
    patch_offsets = np.cumsum([0] + [V.nbasis for V in Vh.spaces])
    dofs, points = [], []
    for bnd in boundaries:
        p = domain.interior.args.index(bnd.domain)
        comp = 1 - bnd.axis                                    # tangential component
        shape = Vh.spaces[p].spaces[comp].coeff_space.npts
        offset = patch_offsets[p] + sum(Vh.spaces[p].spaces[c].nbasis for c in range(comp))
        index = [None, None]
        index[bnd.axis] = 0 if bnd.ext == -1 else shape[bnd.axis] - 1
        index[comp] = np.arange(shape[comp])
        dofs.append(offset + np.ravel_multi_index(tuple(index), shape))

        logical = bnd.domain.logical_domain
        eta = [None, None]
        eta[bnd.axis] = logical.min_coords[bnd.axis] if bnd.ext == -1 else logical.max_coords[bnd.axis]
        for g in Vh.spaces[p].spaces[comp].spaces[comp].greville:
            eta[comp] = g
            points.append(bnd.domain.mapping(*eta))
    return np.concatenate(dofs), np.array(points, dtype=float)

obst_dofs, obst_points = tangential_boundary_dofs(V1h, obstacle_boundaries)

if bc_type == 'pec':
    mask = np.ones(V1h.nbasis)
    mask[obst_dofs] = 0
    P1_hom = (diags(mask) @ cP1.tosparse()).tocsr()
else:
    P1_hom = cP1.tosparse().tocsr()

# primal curl, and transposed curl for the weak Ampere law
C_m   = (D1_m @ P1_hom).tocsr()
CT_H2 = (C_m.T @ H2_m).tocsr()

The incident wave is a plane wave with a smooth front $\chi$ travelling with it,
$$
B_{\rm inc}(\boldsymbol{x}, t) = \chi(t - t_{\rm front} - \boldsymbol{d}\cdot\boldsymbol{x})\,
\big(\cos(k\boldsymbol{d}\cdot\boldsymbol{x})\cos\omega t + \sin(k\boldsymbol{d}\cdot\boldsymbol{x})\sin\omega t\big),
\qquad \boldsymbol{E}_{\rm inc} = (-d_y, d_x)\, B_{\rm inc},
$$
which is an exact solution of Maxwell's equations since $\omega = k$. The front lies on the box boundary at $t = 0$,
and $\chi$ goes smoothly from 0 to 1 over 3 periods.

Without the front, the data vectors on the disks would be combinations of two fixed vectors (cos part, sin part).
Since $\chi$ varies slowly at the scale of the mesh, we keep these two vectors and apply $\chi$ dof by dof, at the dof locations.

In [ ]:
n_periods_ramp = 3
t_front = -min(d[0] * x + d[1] * y for x in box[0] for y in box[1])
xi = np.zeros(V1h.nbasis)
xi[obst_dofs] = obst_points @ d                            # d.x at the dofs on the disks

def ramp(s):
    """ Smooth (C^2) front, from 0 (s <= 0) to 1 (s >= n_periods_ramp periods). """
    s = np.clip(s / (n_periods_ramp * period), 0., 1.)
    return s**3 * (10 - 15 * s + 6 * s**2)

def incident_data(data, t):
    """ Data vector of the incident wave at time t, from its (cos part, sin part). """
    return ramp(t - t_front - xi) * (np.cos(omega * t) * data[0] + np.sin(omega * t) * data[1])

if bc_type == 'pec':
    # lift of -n x E_inc (tangential dofs on the disks of the commuting projection of E_inc)
    lift_data = []
    for f in (np.cos, np.sin):
        E_inc = [lambda x, y, f=f: -d[1] * f(k * (d[0] * x + d[1] * y)),
                 lambda x, y, f=f:  d[0] * f(k * (d[0] * x + d[1] * y))]
        lift_data.append(-(1 - mask) * P1(E_inc).coeffs.toarray())
else:
    # boundary term + int_{disks} B_inc (n x v)
    from sympy import cos, sin
    x, y  = domain.coordinates
    phase = k * (d[0] * x + d[1] * y)
    load_data = []
    for f in (cos, sin):
        l = LinearForm(v, integral(Union(*obstacle_boundaries), f(phase) * cross(nn, v)))
        load_data.append(discretize(l, domain_h, V1h, backend=backend).assemble().toarray())

## Step 6: Time step and patch-wise LU factorization

The time step follows from the largest eigenvalue of the discrete curl-curl operator, with an integer number of steps per period.

With the implicit absorbing term, each step requires solving with $H^1 + \Delta t A_\varepsilon$.
In the broken-FEEC approach this matrix is block-diagonal (one block per patch), so we factor each block once with a sparse LU.

In [ ]:
from scipy.sparse import block_diag
from scipy.sparse.linalg import splu, eigsh

cfl_max = 0.8
lambda_max = eigsh((C_m.T @ H2_m @ C_m).tocsc(), k=1, M=H1_m.tocsc(), which='LA', return_eigenvectors=False)[0]
Nt_per = int(np.ceil(period / (cfl_max * 2 / np.sqrt(lambda_max))))
dt     = period / Nt_per

# the front needs about 8 periods to cross the box, then the solution becomes time-periodic
n_periods, n_periods_avg = 18, 2
Nt = n_periods * Nt_per
print(f'dt = {dt:.4f}, {Nt} time steps')

class PatchwiseLU:
    """ Sparse LU factorization of a block-diagonal multipatch matrix, one patch at a time. """
    def __init__(self, M, patch_dims):
        M = M.tocsc()
        self._offsets = np.cumsum([0] + list(patch_dims))
        blocks = [M[s:e, s:e] for s, e in zip(self._offsets[:-1], self._offsets[1:])]
        assert abs(M - block_diag(blocks)).max() == 0, 'M should be block-diagonal'
        self._lus = [splu(Mk.tocsc()) for Mk in blocks]

    def solve(self, b):
        return np.concatenate([lu.solve(b[s:e]) for lu, s, e in zip(self._lus, self._offsets[:-1], self._offsets[1:])])

H1A_lu = PatchwiseLU(H1_m + dt * A_m, [V.nbasis for V in V1h.spaces])

## Step 7: Time loop

We use the leapfrog (Strang splitting) scheme of `examples/feec/timedomain_maxwell.py`: half a Faraday step, one Ampère step
(implicit in the absorbing term), and another half Faraday step. Over the last periods, we accumulate the discrete Fourier
transform of $B$ to get its complex amplitude
$\hat B = \frac{2}{T}\int B(t)\, e^{i\omega t}\, dt$.
We also store snapshots of $B$ over the first periods, for the movie at the end.

In [ ]:
E_h = np.zeros(V1h.nbasis)
B_h = np.zeros(V2h.nbasis)
B_hat = np.zeros(V2h.nbasis, dtype=complex)
E_lift0 = np.zeros(V1h.nbasis)

# snapshots for the movie: about frames_per_period frames per period, over the first n_periods_movie periods
frames_per_period, n_periods_movie = 5, 12
movie_stride = max(1, Nt_per // frames_per_period)
snapshots = []

for nt in range(Nt):
    t0, t1 = nt * dt, (nt + 1) * dt
    if bc_type == 'pec':
        E_lift1 = incident_data(lift_data, t1)

    # 1/2 Faraday: B^n -> B^{n+1/2}, with the curl of the lift at t^n
    B_h -= (dt/2) * (C_m @ E_h)
    if bc_type == 'pec':
        B_h -= (dt/2) * (D1_m @ E_lift0)

    # Ampere: (H1 + dt A_eps) E^{n+1} = H1 E^n + dt C^T H2 B^{n+1/2} + data
    rhs = H1_m @ E_h + dt * (CT_H2 @ B_h)
    if bc_type == 'pec':
        rhs -= P1_hom.T @ (H1_m @ (E_lift1 - E_lift0))       # time derivative of the lift
    else:
        rhs += dt * (P1_hom.T @ incident_data(load_data, t0 + dt/2))
    E_h = H1A_lu.solve(rhs)

    # 1/2 Faraday: B^{n+1/2} -> B^{n+1}, with the curl of the lift at t^{n+1}
    B_h -= (dt/2) * (C_m @ E_h)
    if bc_type == 'pec':
        B_h -= (dt/2) * (D1_m @ E_lift1)
        E_lift0 = E_lift1

    # running Fourier transform over the last periods
    if nt >= Nt - n_periods_avg * Nt_per:
        B_hat += (2 / (n_periods_avg * Nt_per)) * np.exp(1j * omega * t1) * B_h

    if (nt + 1) <= n_periods_movie * Nt_per and (nt + 1) % movie_stride == 0:
        snapshots.append((t1, B_h.copy()))

## Step 8: Reference solution

mu-diff computes the scattered field with integral equations on the disks. For disks, the same solution can be written
as a sum of outgoing waves around each disk,
$$
u(\boldsymbol{x}) = \sum_p \sum_{|m|\le M_p} c^p_m\, H^{(1)}_m(k r_p)\, e^{i m \theta_p},
$$
whose coefficients are found by writing the boundary condition on each disk in its own Fourier basis, using Graf's addition
theorem for the waves radiated by the other disks.

In [ ]:
from scipy.special import jv, jvp, hankel1, h1vp

centers = np.array([o['center'] for o in obstacles])
radii   = np.array([o['radius'] for o in obstacles])
# PMC: u = 0 (Dirichlet) on the disks, PEC: normal derivative of u = 0 (Neumann)
J = jv      if bc_type == 'pmc' else jvp
H = hankel1 if bc_type == 'pmc' else h1vp

modes   = [np.arange(-M, M+1) for M in np.ceil(k * radii + 4 * (k * radii)**(1/3) + 10).astype(int)]
offsets = np.cumsum([0] + [len(m) for m in modes])
A_ref = np.zeros((offsets[-1], offsets[-1]), dtype=complex)
b_ref = np.zeros(offsets[-1], dtype=complex)
for q_, (Oq, aq, mq) in enumerate(zip(centers, radii, modes)):
    rows = slice(offsets[q_], offsets[q_+1])
    # incident plane wave around O_q (Jacobi-Anger expansion)
    b_ref[rows] = -np.exp(1j * k * d @ Oq) * 1j**mq * np.exp(-1j * mq * theta_inc) * J(mq, k * aq)
    A_ref[rows, rows] = np.diag(H(mq, k * aq))
    for p_, (Op, mp) in enumerate(zip(centers, modes)):
        if p_ != q_:
            # Graf's addition theorem for the wave radiated by disk p, around O_q
            dist, alpha = np.linalg.norm(Oq - Op), np.arctan2((Oq - Op)[1], (Oq - Op)[0])
            nm = mp[None, :] - mq[:, None]
            A_ref[rows, offsets[p_]:offsets[p_+1]] = J(mq, k * aq)[:, None] * hankel1(nm, k * dist) * np.exp(1j * nm * alpha)
c_ref = np.linalg.solve(A_ref, b_ref)

def u_scat_ref(x, y):
    u = np.zeros(np.shape(x), dtype=complex)
    for p_, (Op, mp) in enumerate(zip(centers, modes)):
        r, th = np.hypot(x - Op[0], y - Op[1]), np.arctan2(y - Op[1], x - Op[0])
        for m, c in zip(mp, c_ref[offsets[p_]:offsets[p_+1]]):
            u += c * hankel1(m, k * r) * np.exp(1j * m * th)
    return u

u_inc = lambda x, y: np.exp(1j * k * (d[0] * x + d[1] * y))

## Step 9: Compare the time-harmonic amplitude with the reference

We evaluate $\hat B_h$ (a $V^2_h$ field, pushed forward to the physical domain) on a grid in each patch,
and compute the relative $L^2$ error on the scattered field.

In [ ]:
from psydac.fem.basic import FemField
from psydac.linalg.utilities import array_to_psydac
from psydac.fem.plotting_utilities import get_plotting_grid, get_grid_vals, get_grid_quad_weights

def grid_values(coeffs, etas):
    """ Values of the complex V2h field with the given coefficients on a multipatch grid. """
    re, im = [get_grid_vals(FemField(V2h, coeffs=array_to_psydac(c, V2h.coeff_space)), etas, mappings_list, space_kind='l2')
              for c in (coeffs.real, coeffs.imag)]
    return [r + 1j * i for r, i in zip(re, im)]

# relative L2 error, with a midpoint quadrature in each patch
etas, xx, yy, logvols = get_plotting_grid(mappings, N=20, centered_nodes=True, return_patch_logvols=True)
weights = get_grid_quad_weights(etas, logvols, mappings_list)
u_h   = grid_values(B_hat, etas)
u_ref = [u_scat_ref(x, y) for x, y in zip(xx, yy)]
rel_l2_error = np.sqrt(sum(np.sum(w * abs(a - b)**2) for w, a, b in zip(weights, u_h, u_ref)) /
                       sum(np.sum(w * abs(b)**2) for w, b in zip(weights, u_ref)))
print(f'relative L2 error on the scattered field: {rel_l2_error:.3e}')

The error is limited by the coarse mesh (4 cells per wavelength) and by the first-order absorbing condition, which
partially reflects the scattered waves reaching the box obliquely (about 17% of the amplitude at 45 degrees).
Refining the mesh reduces the error, but reaching a few percent requires a better absorbing layer, such as a PML.

We finally plot the total field, as in mu-diff's near-field maps.

In [ ]:
etas, xx, yy = get_plotting_grid(mappings, N=20)
tot_h   = [a + u_inc(x, y) for a, x, y in zip(grid_values(B_hat, etas), xx, yy)]
tot_ref = [u_scat_ref(x, y) + u_inc(x, y) for x, y in zip(xx, yy)]

fields = {'Re(u_tot), PSYDAC':     [np.real(u) for u in tot_h],
          'Re(u_tot), reference':  [np.real(u) for u in tot_ref],
          '|u_tot - u_tot,ref|':   [abs(a - b) for a, b in zip(tot_h, tot_ref)]}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (label, vals) in zip(axes, fields.items()):
    vmin, vmax = min(v.min() for v in vals), max(v.max() for v in vals)
    for x, y, val in zip(xx, yy, vals):
        im = ax.pcolormesh(x, y, val, vmin=vmin, vmax=vmax, shading='gouraud')
    for o in obstacles:
        ax.add_patch(plt.Circle(o['center'], o['radius'], fill=False, color='k', lw=1.5))
    ax.set_aspect('equal'); ax.set_title(label)
    fig.colorbar(im, ax=ax)
fig.suptitle(f'Total field, {bc_type.upper()} disks, k = {k:.4g}')
plt.show()

## Step 10: Movie of the time evolution

We animate the snapshots stored during the time loop: the scattered field $B_h$ and the total field
$B_h + B_{\rm inc}$, while the front of the incident wave crosses the box from right to left, the disks start
scattering when it reaches them, and the scattered waves fill the box.

To evaluate all the frames at once, we use the vectorized `eval_fields` method of each patch space, followed by the
$L^2$ push-forward (division by the Jacobian determinant of the mapping).

In [ ]:
from matplotlib import animation
from IPython.display import HTML

etas, xx, yy = get_plotting_grid(mappings, N=16)

frames_scat = [[] for _ in snapshots]          # frames_scat[frame][patch]
for p, (V2p, F, eta) in enumerate(zip(V2h.spaces, mappings_list, etas)):
    # Jacobian determinant on the patch grid (computed once)
    det_J = np.array([[np.linalg.det(np.array(F.jacobian(e1, e2), dtype=float)) for e2 in eta[1]] for e1 in eta[0]])
    fields = [FemField(V2h, coeffs=array_to_psydac(B, V2h.coeff_space)).patch_fields[p] for _, B in snapshots]
    for frame, vals in zip(frames_scat, V2p.eval_fields(eta, *fields)):
        frame.append(vals / det_J)

def B_inc(t, x, y):
    # incident plane wave with its travelling front, as in the time loop
    dx = d[0] * x + d[1] * y
    return ramp(t - t_front - dx) * np.cos(k * dx - omega * t)

frames_tot = [[v + B_inc(t, x, y) for v, x, y in zip(frame, xx, yy)] for (t, _), frame in zip(snapshots, frames_scat)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), dpi=60)
quads = []
for ax, frames, label in zip(axes, (frames_scat, frames_tot), ('scattered field $B_h$', 'total field $B_h + B_{inc}$')):
    vmax = max(abs(v).max() for frame in frames for v in frame)
    quads.append([ax.pcolormesh(x, y, v, vmin=-vmax, vmax=vmax, cmap='RdBu_r', shading='gouraud')
                  for x, y, v in zip(xx, yy, frames[0])])
    for o in obstacles:
        ax.add_patch(plt.Circle(o['center'], o['radius'], color='0.3'))
    ax.set_aspect('equal'); ax.set_title(label)
    fig.colorbar(quads[-1][0], ax=ax)
title = fig.suptitle('')

def update(i):
    for qs, frames in zip(quads, (frames_scat, frames_tot)):
        for q, v in zip(qs, frames[i]):
            q.set_array(v.ravel())
    title.set_text(f't = {snapshots[i][0]:.2f}  ({snapshots[i][0] / period:.2f} periods)')
    return [q for qs in quads for q in qs] + [title]

anim = animation.FuncAnimation(fig, update, frames=len(snapshots), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())

### Test the notebook

In [ ]:
import ipytest
ipytest.autoconfig(raise_on_error=True)

In [ ]:
%%ipytest

def test_rel_l2_error():
    assert rel_l2_error < 0.2